In [1]:
import pandas as pd
import numpy as np


In [2]:
import pandas as pd

products = pd.read_csv("Dataset/products.csv")
factors = pd.read_csv("Dataset/emission_factors.csv")
energy = pd.read_csv("Dataset/energy_grid_intensity.csv")


In [3]:
factor_map = factors.set_index(["stage", "type"])["factor"].to_dict()
grid_map = energy.set_index(["metric", "region"])["value"].to_dict()

In [4]:
def calculate_carbon(row):
    
    # Material emissions
    m1 = factor_map[("material", row["material_1"])] * row["unit_weight_kg"] * row["material_share_1"]
    
    m2 = 0
    if pd.notna(row["material_2"]) and row["material_2"] != "":
        m2 = factor_map[("material", row["material_2"])] * row["unit_weight_kg"] * row["material_share_2"]
    
    material_emissions = m1 + m2
    
    # Manufacturing emissions
    manuf_overhead = factor_map[("manufacturing", row["manufacturing_country"])]
    grid_intensity = grid_map[("grid_intensity", row["manufacturing_country"])]
    energy_emissions = row["manufacturing_energy_kwh"] * grid_intensity
    
    manufacturing_emissions = manuf_overhead + energy_emissions
    
    # Transport emissions
    tons = row["unit_weight_kg"] / 1000
    transport_factor = factor_map[("transport", row["transport_mode"])]
    transport_emissions = transport_factor * tons * row["distance_km"]
    
    # Packaging emissions
    packaging_emissions = factor_map[("packaging", row["packaging_type"])]
    
    total = material_emissions + manufacturing_emissions + transport_emissions + packaging_emissions
    
    return total


In [5]:
products["recalculated_co2"] = products.apply(calculate_carbon, axis=1)


In [6]:
print("Mean Absolute Error:",
      np.mean(np.abs(products["recalculated_co2"] - products["co2e_total_kg"])))


Mean Absolute Error: 0.46924198742857115


# STEP 2 — Feature Engineering

In [7]:
# Material carbon intensity feature
def material_intensity(row):
    m1 = factor_map[("material", row["material_1"])] * row["material_share_1"]
    
    m2 = 0
    if pd.notna(row["material_2"]) and row["material_2"] != "":
        m2 = factor_map[("material", row["material_2"])] * row["material_share_2"]
    
    return m1 + m2

products["material_carbon_intensity"] = products.apply(material_intensity, axis=1)

# Manufacturing intensity
products["manufacturing_intensity"] = products["manufacturing_energy_kwh"] * \
    products["manufacturing_country"].map(
        energy.set_index("region")["value"]
    )

# Transport intensity
products["transport_intensity"] = products["distance_km"] * \
    products["transport_mode"].map(
        factors[factors.stage == "transport"].set_index("type")["factor"]
    )


In [8]:
feature_columns = [
    "unit_weight_kg",
    "distance_km",
    "manufacturing_energy_kwh",
    "recycled_content_pct",
    "material_carbon_intensity",
    "manufacturing_intensity",
    "transport_intensity",
    "manufacturing_country",
    "transport_mode",
    "packaging_type",
    "certification"
]

X = products[feature_columns]
y = products["co2e_total_kg"]


In [9]:
X = pd.get_dummies(X, drop_first=True)


# STEP 3 — Train ML Models

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("Linear Regression Results")
print("R2:", r2_score(y_test, y_pred_lr))
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))


Linear Regression Results
R2: 0.6792163453278566
MAE: 4.814963891583522
RMSE: 10.424382177460714


/Users/hetavvyas/ml_env/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/hetavvyas/ml_env/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/hetavvyas/ml_env/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


In [12]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest Results")
print("R2:", r2_score(y_test, y_pred_rf))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))



Random Forest Results
R2: 0.8400365030064046
MAE: 2.7446419285714327
RMSE: 7.361302326487572


In [13]:
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).head(10)

manufacturing_energy_kwh             0.485618
material_carbon_intensity            0.322877
unit_weight_kg                       0.068919
transport_intensity                  0.062060
manufacturing_intensity              0.045399
recycled_content_pct                 0.004886
distance_km                          0.003718
manufacturing_country_china          0.001244
manufacturing_country_south_korea    0.000990
transport_mode_ship                  0.000959
dtype: float64

# Live Prediction Pipeline

In [14]:
def compute_material_carbon_intensity(material_1, share_1, material_2="", share_2=0.0):
    m1 = factor_map[("material", material_1)] * share_1
    m2 = 0.0
    if material_2 and material_2 != "":
        m2 = factor_map[("material", material_2)] * share_2
    return m1 + m2

def compute_manufacturing_intensity(kwh, country):
    grid = grid_map[("grid_intensity", country)]
    return kwh * grid

def compute_transport_intensity(distance_km, transport_mode):
    tf = factor_map[("transport", transport_mode)]
    return distance_km * tf


In [15]:
def make_model_features(new_product_dict, X_columns):
    """
    new_product_dict must contain:
    unit_weight_kg, distance_km, manufacturing_energy_kwh, recycled_content_pct,
    material_1, material_share_1, material_2, material_share_2,
    manufacturing_country, transport_mode, packaging_type, certification
    """
    df = pd.DataFrame([new_product_dict]).copy()

    # engineered features (same logic as training)
    df["material_carbon_intensity"] = df.apply(
        lambda r: compute_material_carbon_intensity(
            r["material_1"], r["material_share_1"], r.get("material_2",""), r.get("material_share_2",0.0)
        ), axis=1
    )

    df["manufacturing_intensity"] = df.apply(
        lambda r: compute_manufacturing_intensity(r["manufacturing_energy_kwh"], r["manufacturing_country"]),
        axis=1
    )

    df["transport_intensity"] = df.apply(
        lambda r: compute_transport_intensity(r["distance_km"], r["transport_mode"]),
        axis=1
    )

    # Select same base columns you trained on (before get_dummies)
    base_cols = [
        "unit_weight_kg", "distance_km", "manufacturing_energy_kwh", "recycled_content_pct",
        "material_carbon_intensity", "manufacturing_intensity", "transport_intensity",
        "manufacturing_country", "transport_mode", "packaging_type", "certification"
    ]
    df = df[base_cols]

    # One-hot encode
    df = pd.get_dummies(df, drop_first=True)

    # Align exactly to training columns
    df = df.reindex(columns=X_columns, fill_value=0)

    return df


In [16]:
def predict_co2_and_score(new_product_dict, model, X_columns, category_for_scoring, products_df):
    X_new = make_model_features(new_product_dict, X_columns)
    pred_co2 = float(model.predict(X_new)[0])

    # category-based scoring (0-100)
    cat_max = products_df[products_df["category"] == category_for_scoring]["co2e_total_kg"].max()
    cat_min = products_df[products_df["category"] == category_for_scoring]["co2e_total_kg"].min()

    # avoid divide by zero
    if cat_max == cat_min:
        score_100 = 50.0
    else:
        # lower CO2 => higher score
        score_100 = 100 * (1 - (pred_co2 - cat_min) / (cat_max - cat_min))
        score_100 = float(np.clip(score_100, 0, 100))

    # letter grade mapping
    if score_100 >= 90: grade = "A"
    elif score_100 >= 80: grade = "B"
    elif score_100 >= 70: grade = "C"
    elif score_100 >= 60: grade = "D"
    elif score_100 >= 50: grade = "E"
    else: grade = "F"

    return pred_co2, score_100, grade


In [17]:
new_product = {
    "unit_weight_kg": 0.22,
    "distance_km": 2200,
    "manufacturing_energy_kwh": 0.8,
    "recycled_content_pct": 60,

    "material_1": "organic_cotton",
    "material_share_1": 0.8,
    "material_2": "recycled_polyester",
    "material_share_2": 0.2,

    "manufacturing_country": "mexico",
    "transport_mode": "rail",
    "packaging_type": "recycled_cardboard_box",
    "certification": "gots"
}

pred_co2, score_100, grade = predict_co2_and_score(
    new_product_dict=new_product,
    model=rf,
    X_columns=X.columns,
    category_for_scoring="t_shirt",
    products_df=products
)

print("Predicted CO2 (kg):", round(pred_co2, 3))
print("Climate Score (0-100):", round(score_100, 1))
print("Grade:", grade)


Predicted CO2 (kg): 2.764
Climate Score (0-100): 91.3
Grade: A


## Alternative Recommendation Engine (Actionable “Reduce Emissions”)

In [18]:
def recommend_greener_version(p):
    p2 = p.copy()

    mode = p2.get("transport_mode")
    if mode == "air":
        p2["transport_mode"] = "ship"
    elif mode == "truck":
        p2["transport_mode"] = "rail"

    pack = p2.get("packaging_type")
    if pack in {"plastic_wrap", "cardboard_box"}:
        p2["packaging_type"] = "recycled_cardboard_box"

    if p2.get("material_1") == "polyester":
        p2["material_1"] = "recycled_polyester"
    if p2.get("material_2") == "polyester":
        p2["material_2"] = "recycled_polyester"

    return p2


def compare_versions(original_product, category):
    co2_orig, score_orig, grade_orig = predict_co2_and_score(
        original_product, rf, X.columns, category, products
    )

    greener_product = recommend_greener_version(original_product)

    co2_green, score_green, grade_green = predict_co2_and_score(
        greener_product, rf, X.columns, category, products
    )

    reduction_pct = 100 * (co2_orig - co2_green) / co2_orig

    return {
        "original_co2": co2_orig,
        "original_score": score_orig,
        "original_grade": grade_orig,
        "greener_co2": co2_green,
        "greener_score": score_green,
        "greener_grade": grade_green,
        "reduction_pct": reduction_pct,
        "greener_product": greener_product
    }


In [19]:
bad_product = new_product.copy()
bad_product["transport_mode"] = "air"
bad_product["packaging_type"] = "plastic_wrap"
bad_product["material_1"] = "polyester"
bad_product["material_share_1"] = 1.0
bad_product["material_2"] = ""
bad_product["material_share_2"] = 0.0

result = compare_versions(bad_product, category="t_shirt")

print("Original CO2:", round(result["original_co2"], 3), "kg | Grade:", result["original_grade"])
print("Greener CO2:", round(result["greener_co2"], 3), "kg | Grade:", result["greener_grade"])
print("Reduction:", round(result["reduction_pct"], 1), "%")

print("\nSuggested Changes:")
for k in ["transport_mode", "packaging_type", "material_1", "material_2"]:
    if bad_product.get(k) != result["greener_product"].get(k):
        print(f"- {k}: {bad_product.get(k)} → {result['greener_product'].get(k)}")


Original CO2: 3.239 kg | Grade: B
Greener CO2: 2.933 kg | Grade: B
Reduction: 9.4 %

Suggested Changes:
- transport_mode: air → ship
- packaging_type: plastic_wrap → recycled_cardboard_box
- material_1: polyester → recycled_polyester


In [20]:
def climatechain_analyze(product_dict, category):
    # predict original
    co2_orig, score_orig, grade_orig = predict_co2_and_score(
        product_dict, rf, X.columns, category, products
    )

    # greener recommendation
    greener = recommend_greener_version(product_dict)
    co2_green, score_green, grade_green = predict_co2_and_score(
        greener, rf, X.columns, category, products
    )

    reduction_pct = 100 * (co2_orig - co2_green) / co2_orig if co2_orig > 0 else 0

    # list the changes
    changes = []
    for k in ["transport_mode", "packaging_type", "material_1", "material_2"]:
        if product_dict.get(k) != greener.get(k):
            changes.append({
                "field": k,
                "from": product_dict.get(k),
                "to": greener.get(k)
            })

    return {
        "original": {
            "predicted_co2_kg": round(co2_orig, 3),
            "score_0_100": round(score_orig, 1),
            "grade": grade_orig
        },
        "recommended": {
            "predicted_co2_kg": round(co2_green, 3),
            "score_0_100": round(score_green, 1),
            "grade": grade_green
        },
        "impact": {
            "reduction_percent": round(reduction_pct, 1)
        },
        "changes": changes
    }


In [21]:
analysis = climatechain_analyze(bad_product, category="t_shirt")
analysis


{'original': {'predicted_co2_kg': 3.239, 'score_0_100': 83.6, 'grade': 'B'},
 'recommended': {'predicted_co2_kg': 2.933, 'score_0_100': 88.6, 'grade': 'B'},
 'impact': {'reduction_percent': 9.4},
 'changes': [{'field': 'transport_mode', 'from': 'air', 'to': 'ship'},
  {'field': 'packaging_type',
   'from': 'plastic_wrap',
   'to': 'recycled_cardboard_box'},
  {'field': 'material_1', 'from': 'polyester', 'to': 'recycled_polyester'}]}

In [22]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Product(BaseModel):
    unit_weight_kg: float
    distance_km: float
    manufacturing_energy_kwh: float
    recycled_content_pct: float
    material_1: str
    material_share_1: float
    material_2: str = ""
    material_share_2: float = 0.0
    manufacturing_country: str
    transport_mode: str
    packaging_type: str
    certification: str

@app.post("/analyze/{category}")
def analyze(category: str, product: Product):
    return climatechain_analyze(product.model_dump(), category)


In [23]:
def carbon_breakdown_engine(p):
    # material
    m1 = factor_map[("material", p["material_1"])] * p["unit_weight_kg"] * p["material_share_1"]
    m2 = 0.0
    if p.get("material_2",""):
        m2 = factor_map[("material", p["material_2"])] * p["unit_weight_kg"] * p["material_share_2"]
    material = m1 + m2

    # manufacturing
    manuf_overhead = factor_map[("manufacturing", p["manufacturing_country"])]
    grid = grid_map[("grid_intensity", p["manufacturing_country"])]
    manuf_energy = p["manufacturing_energy_kwh"] * grid
    manufacturing = manuf_overhead + manuf_energy

    # transport (ton-km)
    tons = p["unit_weight_kg"] / 1000.0
    transport_factor = factor_map[("transport", p["transport_mode"])]
    transport = transport_factor * tons * p["distance_km"]

    # packaging
    packaging = factor_map[("packaging", p["packaging_type"])]

    total = material + manufacturing + transport + packaging

    return {
        "material_kg": round(material, 3),
        "manufacturing_kg": round(manufacturing, 3),
        "transport_kg": round(transport, 3),
        "packaging_kg": round(packaging, 3),
        "total_engine_kg": round(total, 3)
    }


In [24]:
def climatechain_analyze_v2(product_dict, category):
    # ML prediction for totals + grade
    co2_orig, score_orig, grade_orig = predict_co2_and_score(
        product_dict, rf, X.columns, category, products
    )

    greener = recommend_greener_version(product_dict)
    co2_green, score_green, grade_green = predict_co2_and_score(
        greener, rf, X.columns, category, products
    )

    reduction_pct = 100 * (co2_orig - co2_green) / co2_orig if co2_orig > 0 else 0

    # changes list
    changes = []
    for k in ["transport_mode", "packaging_type", "material_1", "material_2"]:
        if product_dict.get(k) != greener.get(k):
            changes.append({"field": k, "from": product_dict.get(k), "to": greener.get(k)})

    # explainability breakdown from engine (transparent)
    breakdown_orig = carbon_breakdown_engine(product_dict)
    breakdown_green = carbon_breakdown_engine(greener)

    # top driver for explanation
    drivers = {k: breakdown_orig[k] for k in ["material_kg","manufacturing_kg","transport_kg","packaging_kg"]}
    top_driver = max(drivers, key=drivers.get)

    return {
        "original": {
            "predicted_co2_kg": round(float(co2_orig), 3),
            "score_0_100": round(float(score_orig), 1),
            "grade": grade_orig,
            "breakdown": breakdown_orig,
            "top_driver": top_driver.replace("_kg","")
        },
        "recommended": {
            "predicted_co2_kg": round(float(co2_green), 3),
            "score_0_100": round(float(score_green), 1),
            "grade": grade_green,
            "breakdown": breakdown_green
        },
        "impact": {"reduction_percent": round(float(reduction_pct), 1)},
        "changes": changes
    }


In [25]:
analysis2 = climatechain_analyze_v2(bad_product, category="t_shirt")
analysis2


{'original': {'predicted_co2_kg': 3.239,
  'score_0_100': 83.6,
  'grade': 'B',
  'breakdown': {'material_kg': 1.21,
   'manufacturing_kg': 1.752,
   'transport_kg': 0.581,
   'packaging_kg': 0.18,
   'total_engine_kg': 3.723},
  'top_driver': 'manufacturing'},
 'recommended': {'predicted_co2_kg': 2.933,
  'score_0_100': 88.6,
  'grade': 'B',
  'breakdown': {'material_kg': 0.66,
   'manufacturing_kg': 1.752,
   'transport_kg': 0.007,
   'packaging_kg': 0.16,
   'total_engine_kg': 2.579}},
 'impact': {'reduction_percent': 9.4},
 'changes': [{'field': 'transport_mode', 'from': 'air', 'to': 'ship'},
  {'field': 'packaging_type',
   'from': 'plastic_wrap',
   'to': 'recycled_cardboard_box'},
  {'field': 'material_1', 'from': 'polyester', 'to': 'recycled_polyester'}]}

## Export Model + Metadata

In [26]:
import joblib
import json


In [27]:
# 1) Save the trained Random Forest model
joblib.dump(rf, "rf_model.pkl")

# 2) Save the exact feature columns used during training
with open("rf_feature_columns.json", "w") as f:
    json.dump(list(X.columns), f)

print("✅ Saved rf_model.pkl and rf_feature_columns.json")


✅ Saved rf_model.pkl and rf_feature_columns.json


In [28]:
import pandas as pd
import numpy as np
import joblib
import json

# Load model + columns
rf = joblib.load("rf_model.pkl")

with open("rf_feature_columns.json", "r") as f:
    FEATURE_COLUMNS = json.load(f)

# Load factor tables
factors = pd.read_csv("Dataset/emission_factors.csv")
energy = pd.read_csv("Dataset/energy_grid_intensity.csv")

factor_map = factors.set_index(["stage", "type"])["factor"].to_dict()
grid_map = energy.set_index(["metric", "region"])["value"].to_dict()


def compute_material_carbon_intensity(material_1, share_1, material_2="", share_2=0.0):
    m1 = factor_map[("material", material_1)] * share_1
    m2 = 0.0
    if material_2 and material_2 != "":
        m2 = factor_map[("material", material_2)] * share_2
    return m1 + m2


def compute_manufacturing_intensity(kwh, country):
    return kwh * grid_map[("grid_intensity", country)]


def compute_transport_intensity(distance_km, transport_mode):
    return distance_km * factor_map[("transport", transport_mode)]


def make_model_features(p: dict):
    df = pd.DataFrame([p]).copy()

    df["material_carbon_intensity"] = df.apply(
        lambda r: compute_material_carbon_intensity(
            r["material_1"], r["material_share_1"],
            r.get("material_2", ""), r.get("material_share_2", 0.0)
        ), axis=1
    )

    df["manufacturing_intensity"] = df.apply(
        lambda r: compute_manufacturing_intensity(r["manufacturing_energy_kwh"], r["manufacturing_country"]),
        axis=1
    )

    df["transport_intensity"] = df.apply(
        lambda r: compute_transport_intensity(r["distance_km"], r["transport_mode"]),
        axis=1
    )

    base_cols = [
        "unit_weight_kg", "distance_km", "manufacturing_energy_kwh", "recycled_content_pct",
        "material_carbon_intensity", "manufacturing_intensity", "transport_intensity",
        "manufacturing_country", "transport_mode", "packaging_type", "certification"
    ]
    df = df[base_cols]

    df = pd.get_dummies(df, drop_first=True)
    df = df.reindex(columns=FEATURE_COLUMNS, fill_value=0)
    return df


def predict_co2(p: dict) -> float:
    X_new = make_model_features(p)
    return float(rf.predict(X_new)[0])

In [29]:
import joblib, json
import pandas as pd

rf_loaded = joblib.load("rf_model.pkl")
cols_loaded = json.load(open("rf_feature_columns.json"))

print("Model loaded:", type(rf_loaded))
print("Num feature columns:", len(cols_loaded))


Model loaded: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
Num feature columns: 26
